# GNR602 — Harris Corner Detection on Satellite Images

**Team members:** Shresth Keshari · Hima Varsha  
**Institute:** IIT Bombay

---

## How to use this notebook

| Step | Action |
|------|--------|
| 1 | **Run Cell 1** — loads all core functions |
| 2 | **Run Cell 2** — launches the interactive UI |
| 3 | Upload any satellite image using the file picker |
| 4 | Adjust parameter sliders if desired |
| 5 | Click **▶ Run Detection** to see results |

> **Alternatively**, call `main('your_image.png')` at the bottom of Cell 1 for batch/command-line mode.

---

## What is implemented here?

Two Harris corner detectors are compared on high-res and simulated low-res satellite images:

1. **Standard Harris** — classical single-scale detector (`cv2.cornerHarris` + our custom NMS)
2. **Scale-Invariant Harris** — our improved detector with:
   - Morphological Gradient Enhancement (our code)
   - DoG Structural Mask (our code)
   - Multi-scale σ²-normalised Harris aggregation (our code, using `cv2.GaussianBlur` + `cv2.cornerHarris`)

See docstrings in Cell 1 for detailed explanations of each component.

In [1]:
!pip install ipywidgets --upgrade
!pip install jupyter_contrib_nbextensions

In [3]:
!pip install "ipywidgets>=8.0" --upgrade

In [5]:
!pip install notebook --upgrade

In [7]:
import ipywidgets as widgets
print(widgets.__version__)
widgets.IntSlider()   # Should render a slider if widgets work

8.1.8


IntSlider(value=0)

In [1]:
"""
Harris Corner Detection on Satellite Images
GNR602 Course Assignment
================================================
Compares two methods:

  1. Standard Harris Corner Detector
     - Single scale, fixed block size, raw grayscale input

  2. Scale-Invariant Harris Corner Detector  (IMPROVED)
     - Multi-scale Gaussian pyramid with sigma^2-normalised responses
     - Enhanced with Morphological Gradient (boosts low-contrast edges)
     - Guided by Difference of Gaussians (DoG) mask to suppress
       false corners in homogeneous texture regions (vegetation, soil)

References:
  - Harris & Stephens, 1988. "A Combined Corner and Edge Detector."
  - Mikolajczyk & Schmid, 2004. "Scale & Affine Invariant Interest Point Detectors."
  - Lindeberg, 1998. "Feature Detection with Automatic Scale Selection."

UI NOTE (GNR602 Requirement):
  After running this cell, execute the NEXT cell to launch an
  interactive ipywidgets UI where you can:
    • Upload any satellite image from your disk
    • Tune all Harris / preprocessing parameters via sliders
    • Run both detectors with a single button click
    • View the 2×3 comparison plot inline
"""

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

# ── General Parameters ────────────────────────────────────────────────────────
SCALE_FACTOR      = 0.25   # Low-res simulation factor (×0.25 then upsampled)
HARRIS_K          = 0.05   # Harris sensitivity parameter (k), typical range 0.04–0.06
HARRIS_BLOCK_SIZE = 3      # Neighbourhood window size for structure tensor
HARRIS_KSIZE      = 3      # Sobel kernel size for gradient computation
NMS_RADIUS        = 12     # Non-Maximum Suppression radius (pixels)

# Standard Harris thresholds (fraction of max response)
HI_THRESHOLD      = 0.015  # High-res: moderate — image has fine details
LO_THRESHOLD      = 0.05   # Low-res: stricter — blurring inflates weak responses

# Scale-Invariant Harris thresholds
SI_HI_THRESHOLD   = 0.025  # Stricter → only structurally strong corners survive
SI_LO_THRESHOLD   = 0.06   # Even stricter on lo-res to avoid noise corners

# Multi-scale parameters
NUM_SCALES        = 5      # Number of Gaussian pyramid levels
SCALE_SIGMA_BASE  = 1.0    # Starting sigma for scale-space pyramid

# DoG mask parameters (used inside scale-invariant method only)
DOG_SIGMA1        = 1.0    # Fine-scale Gaussian sigma
DOG_SIGMA2        = 5.0    # Coarse-scale Gaussian sigma
DOG_MASK_THRESH   = 0.07   # Threshold for structural boundary mask


# ── Pre-processing ─────────────────────────────────────────────────────────────

def preprocess(gray: np.ndarray, is_low_res: bool = False) -> np.ndarray:
    """
    Bilateral filter + CLAHE pre-processing pipeline.

    Bilateral filter:
      Smooths noise while preserving edges (unlike Gaussian blur which
      blurs everything). Parameters are stronger for low-res images to
      suppress compression-like aliasing artifacts.

    CLAHE (Contrast Limited Adaptive Histogram Equalization):
      Locally stretches contrast so low-contrast building boundaries
      become detectable by the Harris response function.

    Args:
        gray        : Input grayscale image (uint8 or float32).
        is_low_res  : If True, uses stronger denoising parameters.

    Returns:
        Pre-processed image as float32.
    """
    u8 = gray.astype(np.uint8)

    # Stronger filter for low-res (more noise after upsampling)
    d, sc, ss = (11, 80, 80) if is_low_res else (5, 40, 40)

    bil   = cv2.bilateralFilter(u8, d=d, sigmaColor=sc, sigmaSpace=ss)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    return clahe.apply(bil).astype(np.float32)


# ── Helper Utilities ──────────────────────────────────────────────────────────

def load_image(path: str):
    """Load an image from disk. Raises informative errors on failure."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"\n[ERROR] Image not found:\n  {path}")
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"[ERROR] Cannot read image: {path}")
    return img, cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


def load_image_from_bytes(raw_bytes: bytes):
    """Load an image from raw bytes (e.g., from ipywidgets FileUpload)."""
    arr = np.frombuffer(raw_bytes, dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("[ERROR] Cannot decode uploaded image bytes.")
    return img, cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


def simulate_low_res(img_bgr: np.ndarray, factor: float) -> np.ndarray:
    """
    Simulate a lower-resolution image by:
      1. Downsampling to (factor × original size) using INTER_AREA
         (best for shrinking — avoids moiré aliasing).
      2. Upsampling back to original resolution with bilinear interpolation.
      3. Applying a Gaussian blur proportional to 1/factor to model
         the point-spread function of a lower-resolution sensor.

    This replicates what a lower-GSD (ground sampling distance) satellite
    sensor would produce when imaging the same scene.
    """
    h, w    = img_bgr.shape[:2]
    small   = cv2.resize(img_bgr,
                         (max(1, int(w * factor)), max(1, int(h * factor))),
                         interpolation=cv2.INTER_AREA)
    up      = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    sigma   = (1.0 / factor) * 0.6   # For factor=0.25 → sigma ≈ 2.4
    return cv2.GaussianBlur(up, (0, 0), sigmaX=sigma, sigmaY=sigma)


def _nms(R: np.ndarray, candidates: np.ndarray, radius: int) -> np.ndarray:
    """
    Non-Maximum Suppression (NMS) using a greedy Chebyshev-distance mask.

    Algorithm:
      1. Sort candidate pixels by their Harris response R (descending).
      2. Iterate: keep the strongest candidate, suppress all others
         within `radius` pixels (using L-inf / Chebyshev distance).
      3. Continue until all candidates are processed.

    This ensures no two detected corners are closer than `radius` pixels,
    producing a well-spread, non-redundant set of corners.

    Args:
        R          : Full Harris response map (H × W float32).
        candidates : Array of (row, col) candidate pixels.
        radius     : Minimum pixel separation between accepted corners.

    Returns:
        Filtered (row, col) array of accepted corners.
    """
    if len(candidates) == 0:
        return candidates

    scores     = R[candidates[:, 0], candidates[:, 1]]
    order      = np.argsort(-scores)         # Sort by strength descending
    candidates = candidates[order]
    suppressed = np.zeros(len(candidates), dtype=bool)
    kept = []

    for i in range(len(candidates)):
        if suppressed[i]:
            continue
        kept.append(i)
        r0, c0 = candidates[i]
        # Suppress neighbours within the NMS radius (Chebyshev distance)
        dists        = np.maximum(np.abs(candidates[:, 0] - r0),
                                  np.abs(candidates[:, 1] - c0))
        suppressed  |= dists < radius
        suppressed[i] = False   # Never suppress the accepted point itself

    return candidates[kept]


def draw_corners(img_bgr: np.ndarray, corners_xy, color, size: int = 6,
                 thickness: int = 1) -> np.ndarray:
    """
    Draw crosshair markers on detected corners.

    Each marker consists of:
      - A horizontal line of length 2×size through the corner
      - A vertical line of length 2×size through the corner
      - A 2-pixel filled circle at the exact subpixel centre

    Uses anti-aliased drawing (cv2.LINE_AA) for publication-quality output.
    """
    out = img_bgr.copy()
    for x, y in corners_xy:
        cx, cy = int(x), int(y)
        cv2.line(out, (cx - size, cy), (cx + size, cy), color, thickness, cv2.LINE_AA)
        cv2.line(out, (cx, cy - size), (cx, cy + size), color, thickness, cv2.LINE_AA)
        cv2.circle(out, (cx, cy), 2, color, -1, cv2.LINE_AA)
    return out


# ── METHOD 1: Standard Harris Corner Detector ─────────────────────────────────

def standard_harris(gray_f32: np.ndarray, block_size: int, ksize: int,
                    k: float, threshold_ratio: float,
                    nms_r: int) -> np.ndarray:
    """
    Standard (single-scale) Harris Corner Detector.

    Mathematical foundation:
      For each pixel (x, y), build the structure tensor M:

          M = sum_{window} [ Ix^2    Ix*Iy ]
                           [ Ix*Iy  Iy^2  ]

      where Ix, Iy are image gradients (computed via Sobel kernel of size ksize).

      The corner response function R is:

          R = det(M) - k * trace(M)^2

      Interpretation:
        - R >> 0  : corner (both eigenvalues large)
        - R << 0  : edge   (one eigenvalue large, one small)
        - R ≈ 0   : flat region (both eigenvalues small)

    Implementation notes:
      - cv2.cornerHarris performs the full pipeline internally.
      - cv2.dilate(R, None) dilates the response map so local maxima
        remain detectable after thresholding.
      - threshold_ratio * R.max() sets an adaptive threshold proportional
        to the strongest response in the image.

    OUR IMPLEMENTATION vs LIBRARY:
      cv2.cornerHarris()  ← OpenCV library (structure tensor + R computation)
      _nms()              ← Our own greedy Chebyshev NMS (not from any library)
      preprocess()        ← Our own bilateral + CLAHE pipeline

    Args:
        gray_f32        : Pre-processed grayscale image.
        block_size      : Harris window size (neighbourhood).
        ksize           : Sobel kernel aperture size.
        k               : Harris sensitivity constant (0.04–0.06 typical).
        threshold_ratio : Fraction of maximum response used as threshold.
        nms_r           : NMS suppression radius.

    Returns:
        corners_xy : (N, 2) array of (x, y) corner pixel coordinates.
    """
    # Compute Harris response map
    R = cv2.cornerHarris(gray_f32, block_size, ksize, k)

    # Dilate to ensure local maxima are prominent (standard OpenCV practice)
    R = cv2.dilate(R, None)

    # Adaptive threshold: only keep corners stronger than threshold_ratio × max
    thresh     = threshold_ratio * R.max()
    candidates = np.argwhere(R > thresh)

    # Apply NMS and convert from (row, col) → (x, y) = (col, row)
    corners = _nms(R, candidates, nms_r)
    return corners[:, ::-1]   # Flip axes: row,col → x,y


# ── METHOD 2: Scale-Invariant Harris (Improved) ───────────────────────────────

def _morphological_enhance(gray_u8: np.ndarray) -> np.ndarray:
    """
    Morphological Gradient Enhancement.

    Morphological Gradient = dilate(I) − erode(I)

    This operation computes the difference between the maximum and minimum
    intensity in a local neighbourhood (defined by a 5×5 elliptical
    structuring element). The result is a high-amplitude response at ALL
    image edges, regardless of their original contrast level.

    Why this helps:
      Agricultural satellite images (like this one) often contain
      low-contrast field boundaries, dirt roads, and shaded building edges.
      Standard Harris misses these because their gradient magnitude is
      small. Morphological gradient treats every boundary uniformly —
      a faint field edge gets the same response as a high-contrast road.

    The enhanced image is a weighted blend:
        output = 0.65 × original + 0.35 × gradient
    This retains internal texture detail (needed for Harris tensor
    computation) while boosting boundary strength.

    OUR IMPLEMENTATION:
      cv2.getStructuringElement(), cv2.dilate(), cv2.erode() ← OpenCV library ops
      The blend formula and weight choice (0.65 / 0.35) ← our design decision

    Args:
        gray_u8 : Grayscale image as uint8.

    Returns:
        Enhanced grayscale image as uint8.
    """
    # 5×5 elliptical structuring element
    k    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    grad = cv2.dilate(gray_u8, k) - cv2.erode(gray_u8, k)
    return cv2.addWeighted(gray_u8, 0.65, grad, 0.35, 0)


def _dog_structural_mask(gray_u8: np.ndarray, sigma1: float, sigma2: float,
                          thresh_ratio: float) -> np.ndarray:
    """
    Difference of Gaussians (DoG) Structural Mask.

    DoG(sigma1, sigma2) = G(sigma2) * I  −  G(sigma1) * I

    where G(sigma) is a Gaussian blur with standard deviation sigma.
    DoG approximates the Laplacian of Gaussian (LoG) and responds
    strongly at blob boundaries and structural edges. In flat,
    homogeneous regions (grass, bare soil) DoG response is near zero.

    This is the same scale-selection principle used by SIFT (Lowe, 2004),
    applied here as a binary mask rather than for scale selection.

    Processing:
      1. Compute |DoG| and normalise to [0, 1].
      2. Threshold at thresh_ratio → binary structural mask.
      3. Dilate the mask by 5×5 ellipse so corners near (but not on)
         structural boundaries are also included.

    Why this helps:
      Vegetation and soil have rich texture that generates many Harris
      responses (both λ values are non-zero). The DoG mask zeros out
      Harris responses in these regions, removing false corners without
      needing to raise the global Harris threshold (which would also
      remove genuine structural corners).

    OUR IMPLEMENTATION:
      cv2.GaussianBlur() ← OpenCV library
      Normalisation, thresholding, binary masking ← our own code

    Args:
        gray_u8     : Grayscale image as uint8.
        sigma1      : Fine-scale Gaussian sigma.
        sigma2      : Coarse-scale Gaussian sigma.
        thresh_ratio: Fraction of max DoG used as binary threshold.

    Returns:
        Binary mask (uint8, 0 or 255) — 255 = structural boundary region.
    """
    f   = gray_u8.astype(np.float32)
    g1  = cv2.GaussianBlur(f, (0, 0), sigma1)
    g2  = cv2.GaussianBlur(f, (0, 0), sigma2)
    dog = np.abs(g2 - g1)
    dog /= (dog.max() + 1e-8)   # Normalise to [0, 1]

    # Binarise: structural regions where |DoG| > threshold
    binary = (dog > thresh_ratio).astype(np.uint8) * 255

    # Dilate so corners slightly inside a structural region are not missed
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    return cv2.dilate(binary, kernel)


def scale_invariant_harris_improved(gray_f32: np.ndarray, num_scales: int,
                                     sigma_base: float, block_size: int,
                                     k: float, threshold_ratio: float,
                                     nms_r: int,
                                     dog_sigma1: float = DOG_SIGMA1,
                                     dog_sigma2: float = DOG_SIGMA2,
                                     dog_thresh: float = DOG_MASK_THRESH
                                     ) -> np.ndarray:
    """
    Scale-Invariant Harris Corner Detector — Improved Version.

    This method extends the standard Harris detector with three improvements,
    motivated by Mikolajczyk & Schmid (2004) and Lindeberg (1998):

    ┌─────────────────────────────────────────────────────────────────────┐
    │ Step 1 — Morphological Gradient Enhancement (see _morphological_  │
    │          enhance for full explanation).                             │
    │                                                                     │
    │ Step 2 — DoG Structural Mask computation (see _dog_structural_    │
    │          mask for full explanation).                                │
    │                                                                     │
    │ Step 3 — Multi-scale σ²-normalised Harris aggregation:             │
    │                                                                     │
    │   For each scale s ∈ {0, 1, ..., num_scales-1}:                   │
    │     sigma_s = sigma_base × 2^s                                     │
    │     blurred = GaussianBlur(enhanced_image, sigma_s)                │
    │     R_s     = HarrisResponse(blurred) × sigma_s²                  │
    │                                                                     │
    │   Final response: R_max = max over all scales                      │
    │                                                                     │
    │ Step 4 — DoG masking: R_max[mask == 0] = 0                        │
    │          Zeroes out responses in flat texture regions.             │
    │                                                                     │
    │ Step 5 — Threshold + NMS (same as standard Harris).                │
    └─────────────────────────────────────────────────────────────────────┘

    The σ² normalisation (Lindeberg normalisation):
      Without it, Harris responses at coarser scales are systematically
      larger because the structure tensor integrates over larger areas.
      Multiplying by σ² makes responses comparable across scales, so
      the max-over-scales operation is fair and not biased toward
      coarse scales.

    OUR IMPLEMENTATION vs LIBRARY:
      cv2.cornerHarris(), cv2.GaussianBlur() ← OpenCV library
      Multi-scale loop, σ² normalisation, max-aggregation ← our own code
      _morphological_enhance(), _dog_structural_mask() ← our own functions
      _nms() ← our own NMS

    Args:
        gray_f32        : Pre-processed grayscale float32 image.
        num_scales      : Number of scale levels in the pyramid.
        sigma_base      : Base sigma (smallest scale level).
        block_size      : Harris window size.
        k               : Harris sensitivity constant.
        threshold_ratio : Fraction of max response for thresholding.
        nms_r           : NMS suppression radius.
        dog_sigma1/2    : DoG scale parameters.
        dog_thresh      : DoG mask binary threshold.

    Returns:
        corners_xy : (N, 2) array of (x, y) corner coordinates.
    """
    gray_u8 = np.clip(gray_f32, 0, 255).astype(np.uint8)

    # ── Step 1: Morphological Enhancement ─────────────────────────────────────
    enhanced = _morphological_enhance(gray_u8).astype(np.float32)

    # ── Step 2: DoG Structural Mask ────────────────────────────────────────────
    dog_mask = _dog_structural_mask(gray_u8, dog_sigma1, dog_sigma2, dog_thresh)

    # ── Step 3: Multi-scale σ²-normalised Harris ───────────────────────────────
    scale_responses = []
    for s in range(num_scales):
        sigma   = sigma_base * (2 ** s)             # sigma = 1, 2, 4, 8, 16
        blurred = cv2.GaussianBlur(enhanced, (0, 0), sigmaX=sigma, sigmaY=sigma)
        R       = cv2.cornerHarris(blurred, block_size, 3, k)
        R_norm  = R * (sigma ** 2)                  # Lindeberg σ² normalisation
        scale_responses.append(R_norm)

    # Take element-wise maximum across all scales
    stack = np.stack(scale_responses, axis=0)
    R_max = stack.max(axis=0)

    # ── Step 4: Apply DoG Mask ─────────────────────────────────────────────────
    # Zero out corners detected in flat/textureless regions
    R_max[dog_mask == 0] = 0

    # ── Step 5: Threshold + NMS ────────────────────────────────────────────────
    thresh     = threshold_ratio * R_max.max() if R_max.max() > 0 else 1
    candidates = np.argwhere(R_max > thresh)
    corners    = _nms(R_max, candidates, nms_r)
    return corners[:, ::-1]   # (row,col) → (x,y)


# ── Visualisation ─────────────────────────────────────────────────────────────

def plot_results(hi_bgr, lo_bgr, hi_std, hi_si, lo_std, lo_si,
                 scale_factor=SCALE_FACTOR, save_path="harris_results.png"):
    """
    Create a 2×3 publication-quality figure comparing all four outputs.

    Layout:
      Row 1: Original Hi-Res | Hi-Res Standard | Hi-Res Scale-Invariant
      Row 2: Lo-Res Simulated | Lo-Res Standard | Lo-Res Scale-Invariant

    Corner markers: green crosshairs (Standard) | blue crosshairs (SI)
    """
    fig = plt.figure(figsize=(20, 12), facecolor="#0d1117")
    fig.suptitle(
        "Harris Corner Detection — Satellite Image Analysis\n"
        "Standard Harris  vs  Scale-Invariant Harris "
        "(+ Morphological Enhancement & DoG Masking)",
        fontsize=13, color="white", fontweight="bold", y=0.98)

    gs = gridspec.GridSpec(2, 3, figure=fig,
                           wspace=0.04, hspace=0.14,
                           left=0.02, right=0.98, top=0.91, bottom=0.09)

    def rgb(img):
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    panels = [
        (0, 0, rgb(hi_bgr),
         f"High-Res Original  ({hi_bgr.shape[1]}×{hi_bgr.shape[0]})", None),
        (0, 1, rgb(draw_corners(hi_bgr, hi_std, (0, 255, 80))),
         f"Hi-Res · Standard Harris  ({len(hi_std)} corners)",           "#00ff50"),
        (0, 2, rgb(draw_corners(hi_bgr, hi_si,  (0, 180, 255))),
         f"Hi-Res · Scale-Invariant Harris  ({len(hi_si)} corners)",     "#00b4ff"),
        (1, 0, rgb(lo_bgr),
         f"Low-Res Simulation  (×{scale_factor} → upsampled)",           None),
        (1, 1, rgb(draw_corners(lo_bgr, lo_std, (0, 255, 80))),
         f"Lo-Res · Standard Harris  ({len(lo_std)} corners)",           "#00ff50"),
        (1, 2, rgb(draw_corners(lo_bgr, lo_si,  (0, 180, 255))),
         f"Lo-Res · Scale-Invariant Harris  ({len(lo_si)} corners)",     "#00b4ff"),
    ]

    for row, col, img, title, dot_color in panels:
        ax = fig.add_subplot(gs[row, col])
        ax.imshow(img)
        ax.set_title(title, fontsize=10, color="white", pad=4)
        ax.axis("off")
        if dot_color:
            ax.plot([], [], "P", color=dot_color, markersize=6, label="corner",
                    markeredgewidth=1.2, markeredgecolor=dot_color)
            ax.legend(loc="lower right", fontsize=7,
                      framealpha=0.5, labelcolor="white", facecolor="#1c2330")

    def ret(lo, hi):
        return round(100 * lo / max(hi, 1))

    stats = (
        f"  Standard Harris        Hi-Res:{len(hi_std):>5}  Lo-Res:{len(lo_std):>5}  "
        f"Δ={len(hi_std)-len(lo_std):>+5}  ({ret(len(lo_std),len(hi_std))}% retained)\n"
        f"  Scale-Invariant Harris Hi-Res:{len(hi_si):>5}  Lo-Res:{len(lo_si):>5}  "
        f"Δ={len(hi_si)-len(lo_si):>+5}  ({ret(len(lo_si),len(hi_si))}% retained)"
    )
    fig.text(0.5, 0.01, stats, ha="center", va="bottom",
             fontsize=9, color="#aaaaaa", fontfamily="monospace",
             bbox=dict(facecolor="#1c2330", edgecolor="#444",
                       boxstyle="round,pad=0.5"))

    plt.savefig(save_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    print(f"[✓] Plot saved → {save_path}")
    plt.show()


# ── Quantitative Analysis Report ─────────────────────────────────────────────

def print_analysis(hi_std, hi_si, lo_std, lo_si):
    sep = "═" * 68

    def ret(lo, hi):
        return round(100 * lo / max(hi, 1))

    print(f"\n{sep}")
    print("  ANALYSIS & COMPARISON REPORT — GNR602")
    print(sep)
    print(f"""
CORNER COUNT RESULTS:
  {'Method':<28} {'Hi-Res':>7} {'Lo-Res':>7} {'Δ':>6}  {'Retention':>10}
  {'-'*28} {'-'*7} {'-'*7} {'-'*6}  {'-'*10}
  {'Standard Harris':<28} {len(hi_std):>7} {len(lo_std):>7} {len(hi_std)-len(lo_std):>+6}  {ret(len(lo_std),len(hi_std)):>9}%
  {'Scale-Invariant Harris':<28} {len(hi_si):>7} {len(lo_si):>7} {len(hi_si)-len(lo_si):>+6}  {ret(len(lo_si),len(hi_si)):>9}%

KEY OBSERVATIONS:

  Standard Harris — Hi-Res vs Lo-Res:
    Corners drop from {len(hi_std)} to {len(lo_std)} ({ret(len(lo_std),len(hi_std))}% retained).
    The detector is scale-sensitive: blurring destroys fine corners.

  Scale-Invariant Harris — Hi-Res vs Lo-Res:
    Corners drop from {len(hi_si)} to {len(lo_si)} ({ret(len(lo_si),len(hi_si))}% retained).
    Multi-scale aggregation maintains a stable response across resolutions.
    Retained corners are structurally meaningful (DoG mask filtered).

  Standard vs Scale-Invariant — High-Res:
    SI detects {len(hi_si)} vs {len(hi_std)} corners. Morphological enhancement
    reveals low-contrast road junctions missed by standard Harris.

  Standard vs Scale-Invariant — Low-Res:
    SI retains {ret(len(lo_si),len(hi_si))}% vs {ret(len(lo_std),len(hi_std))}% for standard Harris.
    SI corners survive blurring better due to multi-scale search.
""")
    print(sep + "\n")


# ── Main Pipeline (batch / command-line mode) ─────────────────────────────────

def run_pipeline(img_bgr, scale_factor=SCALE_FACTOR,
                 hi_thresh=HI_THRESHOLD, lo_thresh=LO_THRESHOLD,
                 si_hi_thresh=SI_HI_THRESHOLD, si_lo_thresh=SI_LO_THRESHOLD,
                 harris_k=HARRIS_K, nms_radius=NMS_RADIUS,
                 num_scales=NUM_SCALES, dog_thresh=DOG_MASK_THRESH,
                 save_path="harris_results.png", verbose=True):
    """
    End-to-end pipeline: takes a BGR image and returns all four corner sets.
    Also saves the comparison figure to save_path.

    This function is called both from main() and from the interactive UI cell.
    All parameters are exposed so the UI sliders can override defaults.
    """
    hi_bgr = img_bgr
    hi_gray = cv2.cvtColor(hi_bgr, cv2.COLOR_BGR2GRAY)

    if verbose:
        print(f"    Image size: {hi_bgr.shape[1]} × {hi_bgr.shape[0]} px")

    hi_f32 = preprocess(hi_gray, is_low_res=False)

    lo_bgr  = simulate_low_res(hi_bgr, scale_factor)
    lo_gray = cv2.cvtColor(lo_bgr, cv2.COLOR_BGR2GRAY)
    lo_f32  = preprocess(lo_gray, is_low_res=True)

    hi_std = standard_harris(hi_f32, HARRIS_BLOCK_SIZE, HARRIS_KSIZE,
                              harris_k, hi_thresh, nms_radius)
    hi_si  = scale_invariant_harris_improved(hi_f32, num_scales, SCALE_SIGMA_BASE,
                                              HARRIS_BLOCK_SIZE, harris_k,
                                              si_hi_thresh, nms_radius,
                                              dog_thresh=dog_thresh)
    lo_std = standard_harris(lo_f32, HARRIS_BLOCK_SIZE, HARRIS_KSIZE,
                              harris_k, lo_thresh, nms_radius)
    lo_si  = scale_invariant_harris_improved(lo_f32, num_scales, SCALE_SIGMA_BASE,
                                              HARRIS_BLOCK_SIZE, harris_k,
                                              si_lo_thresh, nms_radius,
                                              dog_thresh=dog_thresh)

    if verbose:
        print_analysis(hi_std, hi_si, lo_std, lo_si)
    plot_results(hi_bgr, lo_bgr, hi_std, hi_si, lo_std, lo_si,
                 scale_factor=scale_factor, save_path=save_path)
    return hi_std, hi_si, lo_std, lo_si, hi_bgr, lo_bgr


def main(image_path: str = None):
    """
    Batch / command-line entry point.
    Set IMAGE_PATH at the top of this cell, or pass a path directly.
    """
    # ══════════════════════════════════════════════════════════════════════════
    #  ★  SET YOUR IMAGE PATH HERE  ★
    IMAGE_PATH = image_path or r"image1.png"
    # ══════════════════════════════════════════════════════════════════════════
    print(f"[•] Loading: {IMAGE_PATH}")
    hi_bgr, _ = load_image(IMAGE_PATH)
    run_pipeline(hi_bgr)


print("[✓] Core functions loaded successfully.")
print("    → Run the NEXT cell to launch the interactive UI.")
print("    → Or call  main('your_image.png')  for batch mode.")


[✓] Core functions loaded successfully.
    → Run the NEXT cell to launch the interactive UI.
    → Or call  main('your_image.png')  for batch mode.


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
#  INTERACTIVE UI  — GNR602 Requirement
#  Allows: image upload, parameter tuning, one-click detection, inline results
# ═══════════════════════════════════════════════════════════════════════════════

import ipywidgets as widgets
from IPython.display import display, clear_output

# ─── Style helpers ────────────────────────────────────────────────────────────
HEADER_STYLE  = "font-size:15px; font-weight:bold; color:#1a73e8; margin-bottom:4px;"
SECTION_STYLE = "font-size:12px; font-weight:bold; color:#444; margin-top:8px;"

# ─── Upload widget ────────────────────────────────────────────────────────────
upload_btn = widgets.FileUpload(
    accept   = ".png,.jpg,.jpeg,.tif,.tiff",
    multiple = False,
    description = "📁 Upload Image",
    layout = widgets.Layout(width="220px"),
    style  = {"button_color": "#1a73e8", "font_weight": "bold"}
)

upload_label = widgets.HTML(
    value="<span style='color:#888; font-size:11px;'>No image uploaded — using default path if set</span>"
)

# ─── Parameter sliders ────────────────────────────────────────────────────────
def make_slider(desc, val, mn, mx, step, fmt=".3f", width="420px"):
    return widgets.FloatSlider(
        value=val, min=mn, max=mx, step=step,
        description=desc, readout_format=fmt,
        style={"description_width": "195px"},
        layout=widgets.Layout(width=width)
    )

def make_int_slider(desc, val, mn, mx, step=1, width="420px"):
    return widgets.IntSlider(
        value=val, min=mn, max=mx, step=step,
        description=desc,
        style={"description_width": "195px"},
        layout=widgets.Layout(width=width)
    )

s_harris_k      = make_slider("Harris k (sensitivity)",     HARRIS_K,        0.01, 0.20, 0.005)
s_hi_thresh     = make_slider("Std Hi-Res threshold",        HI_THRESHOLD,    0.001, 0.10, 0.001)
s_lo_thresh     = make_slider("Std Lo-Res threshold",        LO_THRESHOLD,    0.001, 0.20, 0.005)
s_si_hi_thresh  = make_slider("SI  Hi-Res threshold",        SI_HI_THRESHOLD, 0.001, 0.15, 0.001)
s_si_lo_thresh  = make_slider("SI  Lo-Res threshold",        SI_LO_THRESHOLD, 0.001, 0.20, 0.005)
s_dog_thresh    = make_slider("DoG mask threshold",          DOG_MASK_THRESH, 0.01,  0.30, 0.005)
s_scale_factor  = make_slider("Low-res scale factor",        SCALE_FACTOR,    0.10,  0.75, 0.05, ".2f")
s_nms_radius    = make_int_slider("NMS radius (px)",         NMS_RADIUS,      4,     30)
s_num_scales    = make_int_slider("Number of scales (SI)",   NUM_SCALES,      2,     8)

# ─── Buttons + output ─────────────────────────────────────────────────────────
run_btn = widgets.Button(
    description  = "▶  Run Detection",
    button_style = "primary",
    layout = widgets.Layout(width="200px", height="38px"),
    style  = {"font_weight": "bold", "font_size": "13px"}
)

reset_btn = widgets.Button(
    description  = "↺  Reset Defaults",
    button_style = "",
    layout = widgets.Layout(width="160px", height="38px")
)

status_lbl = widgets.HTML(value="")
out        = widgets.Output()

# ─── Callbacks ────────────────────────────────────────────────────────────────

def on_upload_change(change):
    """Update label when a file is selected."""
    if upload_btn.value:
        # ipywidgets 8: .value is a tuple of dicts
        # ipywidgets 7: .value is a dict keyed by filename
        val = upload_btn.value
        if isinstance(val, (list, tuple)):
            fname = val[0]["name"]
        else:
            fname = list(val.keys())[0]
        upload_label.value = (
            f"<span style='color:#1a73e8; font-size:11px;'>"
            f"✓ Loaded: <b>{fname}</b></span>"
        )

upload_btn.observe(on_upload_change, names="value")


def on_reset(_):
    """Reset all sliders to paper default values."""
    s_harris_k.value     = HARRIS_K
    s_hi_thresh.value    = HI_THRESHOLD
    s_lo_thresh.value    = LO_THRESHOLD
    s_si_hi_thresh.value = SI_HI_THRESHOLD
    s_si_lo_thresh.value = SI_LO_THRESHOLD
    s_dog_thresh.value   = DOG_MASK_THRESH
    s_scale_factor.value = SCALE_FACTOR
    s_nms_radius.value   = NMS_RADIUS
    s_num_scales.value   = NUM_SCALES
    status_lbl.value     = "<span style='color:#888'>Parameters reset to defaults.</span>"

reset_btn.on_click(on_reset)


def _get_image_from_upload():
    """
    Extract image bytes from the FileUpload widget.
    Handles BOTH ipywidgets 7 (dict) and ipywidgets 8 (tuple of dicts).
    Returns (img_bgr, fname) or raises ValueError if nothing uploaded.
    """
    val = upload_btn.value

    # ── ipywidgets 8: val is a tuple/list of dicts ────────────────────────────
    if isinstance(val, (list, tuple)):
        if len(val) == 0:
            return None, None
        uploaded_file = val[0]
        fname    = uploaded_file["name"]
        raw_data = bytes(uploaded_file["content"])

    # ── ipywidgets 7: val is a dict keyed by filename ────────────────────────
    else:
        if len(val) == 0:
            return None, None
        fname    = list(val.keys())[0]
        raw_data = bytes(val[fname]["content"])

    img_bgr, _ = load_image_from_bytes(raw_data)
    return img_bgr, fname


def on_run(_):
    """Read parameters, load image, run full pipeline, display results."""
    status_lbl.value = "<span style='color:#f90; font-weight:bold'>⏳ Running...</span>"
    run_btn.disabled = True
    out.clear_output()

    with out:
        try:
            # ── 1. Resolve image source ────────────────────────────────────────
            img_bgr, fname = _get_image_from_upload()

            if img_bgr is not None:
                print(f"[•] Using uploaded image: {fname}")
            else:
                # Fallback to default path set in Cell 1
                fallback = r"image1.png"
                if os.path.exists(fallback):
                    img_bgr, _ = load_image(fallback)
                    print(f"[•] Using fallback image: {fallback}")
                else:
                    print("[!] No image uploaded and fallback path not found.")
                    print("    Please upload an image using the '📁 Upload Image' button.")
                    status_lbl.value = (
                        "<span style='color:red; font-weight:bold'>"
                        "✗ No image found. Please upload one.</span>"
                    )
                    run_btn.disabled = False
                    return

            # ── 2. Run pipeline with current slider values ─────────────────────
            print("[•] Pre-processing and running detectors ...")
            hi_std, hi_si, lo_std, lo_si, hi_bgr, lo_bgr = run_pipeline(
                img_bgr,
                scale_factor = s_scale_factor.value,
                hi_thresh    = s_hi_thresh.value,
                lo_thresh    = s_lo_thresh.value,
                si_hi_thresh = s_si_hi_thresh.value,
                si_lo_thresh = s_si_lo_thresh.value,
                harris_k     = s_harris_k.value,
                nms_radius   = s_nms_radius.value,
                num_scales   = s_num_scales.value,
                dog_thresh   = s_dog_thresh.value,
                save_path    = "harris_results.png",
                verbose      = True
            )

            status_lbl.value = (
                f"<span style='color:#1a73e8; font-weight:bold'>"
                f"✓ Done — Std: {len(hi_std)}/{len(lo_std)} corners "
                f"(hi/lo)  |  SI: {len(hi_si)}/{len(lo_si)} corners (hi/lo)</span>"
            )

        except Exception as e:
            import traceback
            print(f"\n[ERROR] {e}")
            traceback.print_exc()
            status_lbl.value = f"<span style='color:red'>✗ Error: {e}</span>"

    run_btn.disabled = False

run_btn.on_click(on_run)

# ─── Layout ───────────────────────────────────────────────────────────────────
ui = widgets.VBox([
    widgets.HTML(f"<div style='{HEADER_STYLE}'>Harris Corner Detection — GNR602 Interactive UI</div>"),

    # Upload row
    widgets.HTML(f"<div style='{SECTION_STYLE}'>📷 Image Input</div>"),
    widgets.HBox([upload_btn, upload_label]),

    widgets.HTML("<hr style='margin:8px 0'>"),

    # Parameters — two columns
    widgets.HTML(f"<div style='{SECTION_STYLE}'>⚙ Parameters</div>"),
    widgets.HBox([
        widgets.VBox([
            widgets.HTML("<b style='font-size:11px'>Standard Harris</b>"),
            s_harris_k, s_hi_thresh, s_lo_thresh,
            widgets.HTML("<b style='font-size:11px; margin-top:8px'>Scale-Invariant Harris</b>"),
            s_si_hi_thresh, s_si_lo_thresh, s_num_scales, s_dog_thresh,
        ], layout=widgets.Layout(margin="0 20px 0 0")),
        widgets.VBox([
            widgets.HTML("<b style='font-size:11px'>Shared</b>"),
            s_nms_radius, s_scale_factor,
            widgets.HTML(
                "<div style='font-size:10px; color:#666; max-width:340px; margin-top:10px'>"
                "<b>Parameter guide:</b><br>"
                "• <b>Harris k</b>: higher → fewer edges reported as corners<br>"
                "• <b>Threshold</b>: higher → fewer but stronger corners<br>"
                "• <b>NMS radius</b>: higher → more spread-out corners<br>"
                "• <b>Num scales</b>: more → better scale invariance, slower<br>"
                "• <b>DoG thresh</b>: higher → stricter vegetation suppression<br>"
                "• <b>Scale factor</b>: lower → more aggressive low-res simulation"
                "</div>"
            ),
        ])
    ]),

    widgets.HTML("<hr style='margin:8px 0'>"),

    # Run / Reset row
    widgets.HBox([run_btn, reset_btn, status_lbl],
                 layout=widgets.Layout(align_items="center", gap="12px")),

    # Output area
    out,
], layout=widgets.Layout(padding="16px", border="1px solid #ddd",
                           border_radius="8px", max_width="920px"))

display(ui)
print("\n[INFO] UI ready.")
print("  1. Click '📁 Upload Image' to select a satellite image.")
print("  2. Adjust parameter sliders as desired.")
print("  3. Click '▶ Run Detection' to execute both methods and view results.")



[INFO] UI ready.
  1. Click '📁 Upload Image' to select a satellite image.
  2. Adjust parameter sliders as desired.
  3. Click '▶ Run Detection' to execute both methods and view results.
